# Brouwer's fixed-point theorem and the game of Hex

Every continuous map of a triangle to itself leaves some point where it is. Colour a point by a
coordinate that the map does not increase; that is a Sperner colouring, and the walk finds a
point that the map barely moves.

In [2]:
from sperner.brouwer import fixed_point


def cities(x):  # who lives where next year, for three cities
    a, b, c = x
    return (
        0.6 * a + 0.28 * b + 0.08 * c,
        0.24 * a + 0.6 * b + 0.32 * c,
        0.16 * a + 0.12 * b + 0.6 * c,
    )


result = fixed_point(cities, 3, tolerance=1e-9)
print("fixed point:", [round(v, 6) for v in result.point])
print(f"moved by {result.residual:.1e}; {result.evaluations} evaluations")

fixed point: [0.336283, 0.40708, 0.256637]
moved by 3.8e-10; 55 evaluations


## Hex never ends in a draw

On a full Hex board exactly one player has a winning chain. The walk along the boundary between
the colours finds it and only looks at the cells it passes.

In [4]:
import random

from sperner.hex import hex_walk, winner

rng = random.Random(3)
k = 11
board = [["H" if rng.random() < 0.5 else "V" for _ in range(k)] for _ in range(k)]
walk = hex_walk(k, lambda cell: board[cell[0]][cell[1]])
print("winner:", walk.winner, "(search agrees:", winner(board) == walk.winner, ")")
print(f"looked at {walk.looked_at} of {k * k} cells; chain of {len(walk.chain)} cells")

winner: V (search agrees: True )
looked at 35 of 121 cells; chain of 13 cells


## Gale: Hex proves Brouwer

Colour grid points of the square by the direction in which a map pushes them. If no point were
nearly fixed, one player would win Hex — and a winning chain would need two neighbours pushed in
opposite directions, which continuity forbids on a fine board.

In [6]:
import math

from sperner.hex import gale_fixed_point


def turn(z):  # turn the square by 2 radians around its centre and shrink it
    x, y = z[0] - 0.5, z[1] - 0.5
    return (
        0.5 + 0.7 * (math.cos(2) * x - math.sin(2) * y),
        0.5 + 0.7 * (math.sin(2) * x + math.cos(2) * y),
    )


for eps in (1e-2, 1e-3, 1e-4):
    r = gale_fixed_point(turn, eps)
    point = f"{r.point[0]:.4f}, {r.point[1]:.4f}"
    print(f"eps {eps:g}: point {point}; board {r.k}; {r.evaluations} evaluations")

eps 0.01: point 0.4961, 0.4961; board 128; 467 evaluations
eps 0.001: point 0.4995, 0.4995; board 1024; 2803 evaluations
eps 0.0001: point 0.5000, 0.4999; board 16384; 61399 evaluations


## Exercises

1. Compare the evaluations of `gale_fixed_point` and `fixed_point` for the same precision. Why
   does one grow like 1/eps and the other hardly at all?
2. Find a map of the triangle to itself with no fixed point that is not continuous.
3. Show that on a square board where squares touch only along edges, a draw is possible.
4. Why can a winning chain from the west edge in Gale's colouring not start with H−?